## Build me a map, ChatGPT

Build me a run club map from this website:
https://docs.google.com/spreadsheets/d/1QFRce-siqNQsuTEtmCf4zg5uE0W4fTPdJjEDbDVNRLQ/edit?gid=1234181538#gid=1234181538 

In [1]:
import pandas as pd
url = 'https://docs.google.com/spreadsheets/d/1QFRce-siqNQsuTEtmCf4zg5uE0W4fTPdJjEDbDVNRLQ/edit?usp=sharing'
sheet_id = url.split('/d/')[1].split('/edit')[0]
export_url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=274776461'
df = pd.read_csv(export_url)
df.head()


,Borough,Crew,Status,Website,Day,Time,Location,Type of Run,Distance/Run Type,Pace,Meet Up Information,Recognized Organization
0,The Bronx,718Run,Active,https://718run.com/,Thursday,6:30 PM,"Kingbridge Social Club, 3625 Kingsbridge Ave, ...",NaN,NaN,NaN,Free,NaN
1,The Bronx,Boogie Down Bronx Runners,Active,https://www.instagram.com/boogiedownbronxrunners/,Tuesday,7:00 PM,Pelham Cornerstone Community Center - 785 Pelh...,NaN,NaN,NaN,"Free, all levels",NaN
2,The Bronx,Bronx Burners,Active,https://www.instagram.com/bronxburnersrc/,Tuesday,7:00 PM,"The Armory, 216 Fort Washington Ave, New York,...",Track,NaN,NaN,"Free, all levels, winter meetups at the Armory...",NYRR
3,The Bronx,Bronx Femme Run,Active,https://www.instagram.com/bronxfemmerun/,Friday,6:30 PM,"Joseph Yancey Track, East 161st St. &, Macombs...",NaN,"Walk 2.5M, Run/Walk 3.2M, Run 3.5M",NaN,"Every 1st Friday, Free, Womxn club",NaN
4,The Bronx,Bronx Nomads,Active,https://www.instagram.com/bronxnomads/,Monday,6:30 PM,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",NaN,Community Run,NaN,"Free, all levels",NaN


In [187]:
#bronx = df[df['Borough'] == 'Manhattan']
#bronx.shape


In [2]:
print(df.shape)
#df = bronx

#Preprocessing
df_preprocessed = df[df['Status'] == 'Active'] #limiting to active
df_preprocessed = df_preprocessed[df_preprocessed['Location'].notna()] #Limiting to w/ location since it is a map
df_preprocessed = df_preprocessed[~df_preprocessed.Location.str.contains("Varies|social|instagram", case=False, na=False)]
df_preprocessed['Borough'] = df['Borough'].str.replace('The Bronx', 'Bronx') #Cleaning up borough for consistency
df_preprocessed['Weekend'] = df_preprocessed['Day'].isin(['Saturday', 'Sunday']) #Adding weekend indicator
df_preprocessed = df_preprocessed.drop_duplicates() #Drop obvious duplicates

print(df_preprocessed.shape)
df_preprocessed = df_preprocessed.reset_index(drop=True)
df_preprocessed.head()

(425, 12)
(301, 13)


,Borough,Crew,Status,Website,Day,Time,Location,Type of Run,Distance/Run Type,Pace,Meet Up Information,Recognized Organization,Weekend
0,Bronx,Boogie Down Bronx Runners,Active,https://www.instagram.com/boogiedownbronxrunners/,Tuesday,7:00 PM,Pelham Cornerstone Community Center - 785 Pelh...,NaN,NaN,NaN,"Free, all levels",NaN,False
1,Bronx,Bronx Burners,Active,https://www.instagram.com/bronxburnersrc/,Tuesday,7:00 PM,"The Armory, 216 Fort Washington Ave, New York,...",Track,NaN,NaN,"Free, all levels, winter meetups at the Armory...",NYRR,False
2,Bronx,Bronx Femme Run,Active,https://www.instagram.com/bronxfemmerun/,Friday,6:30 PM,"Joseph Yancey Track, East 161st St. &, Macombs...",NaN,"Walk 2.5M, Run/Walk 3.2M, Run 3.5M",NaN,"Every 1st Friday, Free, Womxn club",NaN,False
3,Bronx,Bronx Nomads,Active,https://www.instagram.com/bronxnomads/,Monday,6:30 PM,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",NaN,Community Run,NaN,"Free, all levels",NaN,False
4,Bronx,Bronx Nomads,Active,https://www.instagram.com/bronxnomads/,Thursday,6:30 PM,"Williamsbridge Oval Track, 2335 Resevoir Oval ...",NaN,Track,NaN,"Free, all levels",NaN,False


In [3]:
unique_clubs = df_preprocessed[['Crew', 'Borough', 'Website', 'Location']].drop_duplicates()
unique_clubs

,Crew,Borough,Website,Location
0,Boogie Down Bronx Runners,Bronx,https://www.instagram.com/boogiedownbronxrunners/,Pelham Cornerstone Community Center - 785 Pelh...
1,Bronx Burners,Bronx,https://www.instagram.com/bronxburnersrc/,"The Armory, 216 Fort Washington Ave, New York,..."
2,Bronx Femme Run,Bronx,https://www.instagram.com/bronxfemmerun/,"Joseph Yancey Track, East 161st St. &, Macombs..."
3,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Park, 2335 Resevoir Oval E..."
4,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Track, 2335 Resevoir Oval ..."
...,...,...,...,...
293,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Clove Lakes Park
294,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Silver Lake Park
296,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Greenbelt Nature Center
298,The Just Us Running Club,Staten Island,https://www.instagram.com/thejurcnyc/,Jus' Us Store


## GEOCODE

*Prompt*  

Prompt 1 Chain-of-Thought

I want to geo code addresses using the google maps api. The addresses are stored in the column 'location'. I want to send the api the addresses and then get the lat and long as new columns in my df. I also want to return the status of the api call and the formatted address from the api. 

Prompt 2 Chain-of-Thought
Add borough-specific biasing for better results if first attempt of standard geocoding fails. (Claude automatically came up with third attempt: fuzzy matching.)

Get my API keys

In [4]:
def geocode_address(address, borough=None):
    """Geocode an address using Google Maps API with fallback for approximate matches."""
    try:
        # First attempt: standard geocoding
        response = gmaps.geocode(address)
        
        if response:
            location = response[0]['geometry']['location']
            formatted_address = response[0]['formatted_address']
            result_type = response[0]['types'][0] if 'types' in response[0] else 'unknown'
            return location['lat'], location['lng'], "OK", formatted_address, result_type
            
        # Second attempt: add borough-specific biasing for better results
        if borough:
            # Create a more specific search string with the borough
            borough_search = f"{address}, {borough}, New York City"
            response = gmaps.geocode(borough_search)
            
            if response:
                location = response[0]['geometry']['location']
                formatted_address = response[0]['formatted_address']
                result_type = response[0]['types'][0] if 'types' in response[0] else 'unknown'
                return location['lat'], location['lng'], "BOROUGH_MATCH", formatted_address, result_type
            
        # THIRD attempt: fuzzy matching with geocode places API
        places_response = gmaps.places(address + " New York City")
        
        if places_response['results']:
            place = places_response['results'][0]
            location = place['geometry']['location']
            formatted_address = place['formatted_address'] if 'formatted_address' in place else place['name']
            return location['lat'], location['lng'], "FUZZY_MATCH", formatted_address, "place"
            
        return None, None, "ZERO_RESULTS", None, None
        
    except Exception as e:
        return None, None, "ERROR", str(e), None

In [5]:
#Applying geocode function to preprocessed df
unique_clubs[['latitude', 'longitude', 'api_status', 'formatted_address', 'result_type']] = unique_clubs.apply(
    lambda row: pd.Series(geocode_address(row['Location'], row['Borough'])), 
    axis=1
)

In [6]:
unique_clubs

,Crew,Borough,Website,Location,latitude,longitude,api_status,formatted_address,result_type
0,Boogie Down Bronx Runners,Bronx,https://www.instagram.com/boogiedownbronxrunners/,Pelham Cornerstone Community Center - 785 Pelh...,None,None,ERROR,name 'gmaps' is not defined,None
1,Bronx Burners,Bronx,https://www.instagram.com/bronxburnersrc/,"The Armory, 216 Fort Washington Ave, New York,...",None,None,ERROR,name 'gmaps' is not defined,None
2,Bronx Femme Run,Bronx,https://www.instagram.com/bronxfemmerun/,"Joseph Yancey Track, East 161st St. &, Macombs...",None,None,ERROR,name 'gmaps' is not defined,None
3,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",None,None,ERROR,name 'gmaps' is not defined,None
4,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Track, 2335 Resevoir Oval ...",None,None,ERROR,name 'gmaps' is not defined,None
...,...,...,...,...,...,...,...,...,...
293,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Clove Lakes Park,None,None,ERROR,name 'gmaps' is not defined,None
294,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Silver Lake Park,None,None,ERROR,name 'gmaps' is not defined,None
296,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Greenbelt Nature Center,None,None,ERROR,name 'gmaps' is not defined,None
298,The Just Us Running Club,Staten Island,https://www.instagram.com/thejurcnyc/,Jus' Us Store,None,None,ERROR,name 'gmaps' is not defined,None


## Manually verifying addresses that did not have 'OK' api_status 

In [7]:
unique_clubs_error = unique_clubs[unique_clubs['api_status'] != 'OK']
unique_clubs_error

,Crew,Borough,Website,Location,latitude,longitude,api_status,formatted_address,result_type
0,Boogie Down Bronx Runners,Bronx,https://www.instagram.com/boogiedownbronxrunners/,Pelham Cornerstone Community Center - 785 Pelh...,None,None,ERROR,name 'gmaps' is not defined,None
1,Bronx Burners,Bronx,https://www.instagram.com/bronxburnersrc/,"The Armory, 216 Fort Washington Ave, New York,...",None,None,ERROR,name 'gmaps' is not defined,None
2,Bronx Femme Run,Bronx,https://www.instagram.com/bronxfemmerun/,"Joseph Yancey Track, East 161st St. &, Macombs...",None,None,ERROR,name 'gmaps' is not defined,None
3,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",None,None,ERROR,name 'gmaps' is not defined,None
4,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Track, 2335 Resevoir Oval ...",None,None,ERROR,name 'gmaps' is not defined,None
...,...,...,...,...,...,...,...,...,...
293,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Clove Lakes Park,None,None,ERROR,name 'gmaps' is not defined,None
294,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Silver Lake Park,None,None,ERROR,name 'gmaps' is not defined,None
296,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Greenbelt Nature Center,None,None,ERROR,name 'gmaps' is not defined,None
298,The Just Us Running Club,Staten Island,https://www.instagram.com/thejurcnyc/,Jus' Us Store,None,None,ERROR,name 'gmaps' is not defined,None


In [ ]:
#Checking wrong locations
#print(unique_clubs_error.loc[147])
#print("\nLocation:" , unique_clubs_error.loc[147]['Location'])


In [8]:
#Manually corrected addresses for four run clubs where meetup descriptions required special attention
unique_clubs_error.loc[77, 'Location'] = 'Grand Army Plaza Entrance, Brooklyn'
unique_clubs_error.loc[147, 'Location'] = '910 5th Ave, New York'
unique_clubs_error.loc[185, 'Location'] = '2 W 77th St, New York'
unique_clubs_error.loc[229, 'Location'] = '42 W 14th St, New York'
#Applying geocode function to preprocessed df
unique_clubs_error[['latitude', 'longitude', 'api_status', 'formatted_address', 'result_type']] = unique_clubs_error.apply(
    lambda row: pd.Series(geocode_address(row['Location'], row['Borough'])), 
    axis=1
)
unique_clubs_error

,Crew,Borough,Website,Location,latitude,longitude,api_status,formatted_address,result_type
0,Boogie Down Bronx Runners,Bronx,https://www.instagram.com/boogiedownbronxrunners/,Pelham Cornerstone Community Center - 785 Pelh...,None,None,ERROR,name 'gmaps' is not defined,None
1,Bronx Burners,Bronx,https://www.instagram.com/bronxburnersrc/,"The Armory, 216 Fort Washington Ave, New York,...",None,None,ERROR,name 'gmaps' is not defined,None
2,Bronx Femme Run,Bronx,https://www.instagram.com/bronxfemmerun/,"Joseph Yancey Track, East 161st St. &, Macombs...",None,None,ERROR,name 'gmaps' is not defined,None
3,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",None,None,ERROR,name 'gmaps' is not defined,None
4,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Track, 2335 Resevoir Oval ...",None,None,ERROR,name 'gmaps' is not defined,None
...,...,...,...,...,...,...,...,...,...
293,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Clove Lakes Park,None,None,ERROR,name 'gmaps' is not defined,None
294,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Silver Lake Park,None,None,ERROR,name 'gmaps' is not defined,None
296,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Greenbelt Nature Center,None,None,ERROR,name 'gmaps' is not defined,None
298,The Just Us Running Club,Staten Island,https://www.instagram.com/thejurcnyc/,Jus' Us Store,None,None,ERROR,name 'gmaps' is not defined,None


In [9]:
unique_clubs_error['api_status'] = 'OK'
unique_clubs.update(unique_clubs_error[['result_type', 'api_status' ,'latitude', 'longitude']])

In [10]:
final_clubs = unique_clubs.copy()
final_clubs = final_clubs.drop(298).drop(231)
final_clubs #final list of clubs

,Crew,Borough,Website,Location,latitude,longitude,api_status,formatted_address,result_type
0,Boogie Down Bronx Runners,Bronx,https://www.instagram.com/boogiedownbronxrunners/,Pelham Cornerstone Community Center - 785 Pelh...,None,None,OK,name 'gmaps' is not defined,None
1,Bronx Burners,Bronx,https://www.instagram.com/bronxburnersrc/,"The Armory, 216 Fort Washington Ave, New York,...",None,None,OK,name 'gmaps' is not defined,None
2,Bronx Femme Run,Bronx,https://www.instagram.com/bronxfemmerun/,"Joseph Yancey Track, East 161st St. &, Macombs...",None,None,OK,name 'gmaps' is not defined,None
3,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",None,None,OK,name 'gmaps' is not defined,None
4,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Track, 2335 Resevoir Oval ...",None,None,OK,name 'gmaps' is not defined,None
...,...,...,...,...,...,...,...,...,...
292,Richmond Rockets,Staten Island,https://www.therichmondrockets.com/,Clove Lakes Park,None,None,OK,name 'gmaps' is not defined,None
293,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Clove Lakes Park,None,None,OK,name 'gmaps' is not defined,None
294,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Silver Lake Park,None,None,OK,name 'gmaps' is not defined,None
296,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Greenbelt Nature Center,None,None,OK,name 'gmaps' is not defined,None


In [11]:
final_clubs[final_clubs['Crew'] == 'Rabbit Movers Running Club']
final_clubs

,Crew,Borough,Website,Location,latitude,longitude,api_status,formatted_address,result_type
0,Boogie Down Bronx Runners,Bronx,https://www.instagram.com/boogiedownbronxrunners/,Pelham Cornerstone Community Center - 785 Pelh...,None,None,OK,name 'gmaps' is not defined,None
1,Bronx Burners,Bronx,https://www.instagram.com/bronxburnersrc/,"The Armory, 216 Fort Washington Ave, New York,...",None,None,OK,name 'gmaps' is not defined,None
2,Bronx Femme Run,Bronx,https://www.instagram.com/bronxfemmerun/,"Joseph Yancey Track, East 161st St. &, Macombs...",None,None,OK,name 'gmaps' is not defined,None
3,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Park, 2335 Resevoir Oval E...",None,None,OK,name 'gmaps' is not defined,None
4,Bronx Nomads,Bronx,https://www.instagram.com/bronxnomads/,"Williamsbridge Oval Track, 2335 Resevoir Oval ...",None,None,OK,name 'gmaps' is not defined,None
...,...,...,...,...,...,...,...,...,...
292,Richmond Rockets,Staten Island,https://www.therichmondrockets.com/,Clove Lakes Park,None,None,OK,name 'gmaps' is not defined,None
293,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Clove Lakes Park,None,None,OK,name 'gmaps' is not defined,None
294,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Silver Lake Park,None,None,OK,name 'gmaps' is not defined,None
296,Staten Island Athletic Club,Staten Island,https://www.instagram.com/staten_island_athlet...,Greenbelt Nature Center,None,None,OK,name 'gmaps' is not defined,None


In [ ]:
df_preprocessed
grouped_df = df_preprocessed.groupby(['Borough', 'Crew', 'Location']).agg(
    Days=('Day', lambda x: list(x)),
    Times=('Time', lambda x: list(x)),
    Weekend=('Weekend', lambda x: list(x))
).reset_index()

def classify_day_type(day_list):
    if all(day_list):  # all True
        return 'Weekend'
    elif any(day_list):  # mix of True/False
        return 'Both'
    else:  # all False
        return 'Weekday'

# Preview the result
grouped_df['DayType'] = grouped_df['Weekend'].apply(classify_day_type)
grouped_df[grouped_df['Times'].apply(lambda x: len(x) >= 2 if isinstance(x, list) else False)]

,Borough,Crew,Location,Days,Times,Weekend,DayType
8,Bronx,Bronx Sole,"Franz Sigel Park, E. 153rd Street & Grand Conc...","[Tuesday, Saturday]","[6:15 PM, 10:00 AM]","[False, True]",Both
15,Bronx,Van Cortlandt Track Club,Van Cortland Park - West 242nd Street and Broa...,"[Tuesday, Saturday]","[7:00 PM, 8:00 AM]","[False, True]",Both
18,Brooklyn,BK Heights Run Club,Borough Hall,"[Monday, Tuesday, Wednesday, Thursday, Friday,...","[6:30 PM, 6:30 PM, 6:30 PM, 6:30 PM, 6:30 PM, ...","[False, False, False, False, False, True]",Both
20,Brooklyn,BK Heights Run Club,Under BK Bridge,"[Monday, Tuesday, Wednesday, Thursday, Friday]","[7:20 AM, 7:20 AM, 7:20 AM, 7:20 AM, 7:20 AM]","[False, False, False, False, False]",Weekday
22,Brooklyn,BK Run Project,Prospect Park - Grand Army Plaza entrance,"[Tuesday, Thursday]","[6:30 AM, 6:30 AM]","[False, False]",Weekday
26,Brooklyn,Black Men Run,Prospect Park - Grand Army Plaza entrance,"[Wednesday, Saturday, Weekend]","[7:15 PM, 8:00 AM, nan]","[False, True, False]",Both
28,Brooklyn,Brooklyn Rogues,Prospect Park (Grand Army Plaza),"[Tuesday, Thursday]","[6:00 AM, 6:00 AM]","[False, False]",Weekday
30,Brooklyn,Brooklyn Track Club,East River Track,"[Tuesday, Tuesday]","[7:00 AM, 7:00 PM]","[False, False]",Weekday
31,Brooklyn,Brooklyn Track Club,McCarren Park Track,"[Tuesday, Tuesday]","[7:00 AM, 7:00 PM]","[False, False]",Weekday
35,Brooklyn,Crown Heights Running Club,Prospect Park - Eastern Parkway & Franklin Ave,"[Friday, Saturday]","[6:30 AM, 8:00 AM]","[False, True]",Both


In [258]:
unique_lists = grouped_df['Times'].dropna().apply(tuple).unique()
for item in unique_lists:
    print(list(item))


['Paused']
['8:00 AM']
['7:00 PM']
['6:30 PM']
['6:15 PM', '10:00 AM']
['6:00 PM']
['7:30 PM']
['7:00 PM', '8:00 AM']
['7:00 AM']
['6:30 PM', '6:30 PM', '6:30 PM', '6:30 PM', '6:30 PM', '8:00 AM']
['9:00 AM']
['7:20 AM', '7:20 AM', '7:20 AM', '7:20 AM', '7:20 AM']
['Varies - check website']
['6:30 AM', '6:30 AM']
['6:15 AM']
['6:30 AM']
['7:15 PM', '8:00 AM', nan]
['7:30 AM']
['6:00 AM', '6:00 AM']
[nan]
['7:00 AM', '7:00 PM']
['6:30 AM', '8:00 AM']
['7:00 PM', '8:30 AM']
['6:45 AM']
['7:15 AM']
['7:00 PM', '7:00 PM']
['6:30 PM', '6:30 PM']
['6:00 PM', '8:00 AM']
['8:30 AM']
['6:45 PM']
['6:45 AM', '7:30 PM', '9:10 PM', '6:30 AM', '7:30 PM', '6:00 AM', '7:00 PM', '6:30 AM', '7:30 PM', '8:00 AM', '9:00 AM', '8:00 AM']
['7:00 PM', '7:30 PM', '6:30 PM']
['6:28 AM']
['6:30 PM', '10:00 AM']
['7:00 PM', '9:00 AM']
['7:00 PM', '7:30 AM', '7:00 PM', '10:00 AM', '11:00 AM']
['6:50 PM']
['7:00 AM', '7:00 AM']
['8:00 PM']
['7:00 PM', '7:00 PM', '9:45 AM']
['6:45 AM', '7:15 PM', '7:00 PM', '6:45 A

## Create Interactive Google Map

https://www.iconicsandwiches.com/

**Task:** 

Generate an HTML template that displays an interactive Google Map with custom sandwich markers. When a marker is clicked, a Bootstrap modal should appear, showing sandwich details (restaurant name, sandwich name, price, description, image, location, website link, and a "Get Directions" link).

**Requirements:**

- Bootstrap for styling, including a fixed-top navbar and a fixed-bottom footer.
- Google Maps API for interactive map functionality.
- Custom Markers: Each marker should use a sandwich image as its icon.
- Bootstrap Modal: The modal should display:
   - Sandwich Name
   - Restaurant Name
   - Price
   - Description
   - Sandwich Image
   - Restaurant Location
   - Clickable Website Link
   - "Get Directions" link to Google Maps

**Data Structure:** 

The list of sandwich locations should be stored in an array of JavaScript objects. Each object should include:

{  
   'sandwich_name': 'Chorizo egg sandwich',  
   'restaurant': 'C&B',  
   'lat': 40.7250765,  
   'lng': -73.9816951,  
   'img': 'https://static01.nytimes.com/newsgraphics/2024-05-21-sandwich/_images/breakfast2-@@-300.webp',  
   'address': '178 East Seventh Street (Avenue B)',  
   'price': 13,  
   'description': 'Many people wouldn’t guess that C & B, .....',  
   'website': 'candbnyc.com'  
}

**Event Handling:** 

Clicking a marker should populate the modal with data from the corresponding object and display the modal.

**Additional Styling & Functionality:**

- The map should take up 100% width and full viewport height (100vh).
- The modal should be centered and mobile-friendly (modal-lg).
- Use Bootstrap utility classes for alignment and spacing.
- Include jQuery and Bootstrap scripts for modal functionality.

**Expected Output:** 

A full HTML, CSS, and JavaScript template that meets all the above specifications.


#### print python dictionaries for google map

In [ ]:
locations = []
for index, row in final_clubs.iterrows():
    location = {
        'RunClubName': row['Crew'],
        'Borough': row['Borough'],
        'Address': row['formatted_address'],
        'Website': row['Website']
    }
    locations.append(location)
locations

In [239]:
locations

[{'RunClubName': 'Boogie Down Bronx Runners',
  'Borough': 'Bronx',
  'Address': '785 Pelham Pkwy N, Bronx, NY 10467, USA',
  'Website': 'https://www.instagram.com/boogiedownbronxrunners/'},
 {'RunClubName': 'Bronx Burners',
  'Borough': 'Bronx',
  'Address': '216 Fort Washington Ave, New York, NY 10032, USA',
  'Website': 'https://www.instagram.com/bronxburnersrc/'},
 {'RunClubName': 'Bronx Femme Run',
  'Borough': 'Bronx',
  'Address': 'East 161st St. &, Macombs Dam Bridge, Bronx, NY 10451, USA',
  'Website': 'https://www.instagram.com/bronxfemmerun/'},
 {'RunClubName': 'Bronx Nomads',
  'Borough': 'Bronx',
  'Address': 'Holt Place and, Reservoir Oval E, Bronx, NY 10467, USA',
  'Website': 'https://www.instagram.com/bronxnomads/'},
 {'RunClubName': 'Bronx Nomads',
  'Borough': 'Bronx',
  'Address': '2335 Reservoir Oval E, Bronx, NY 10467, USA',
  'Website': 'https://www.instagram.com/bronxnomads/'},
 {'RunClubName': 'Bronx Rockets',
  'Borough': 'Bronx',
  'Address': 'Rev. T. Wendell

In [ ]:
# locations 

## Remove backgrounds from images

This was a pita pit of despair. ChatGPT led my to trying several python packages I could not install. I updated my python install. Still, nothing. That is until I discovered the image pig api on reddit.

https://imagepig.com/docs/#cutout

<br>

*Prompt 0 Chain-of-Thought*

I need to use the api below to remove the background from images. The image URLs are stored in a df column.

```python
from imagepig import ImagePig

imagepig = ImagePig("your-api-key")
result = imagepig.cutout("https://imagepig.com/static/street.jpeg")
result.save("cutout.png")
```

<br>

*Prompt 1 Chain-of-Thought*

Write the images to an 'output_dir'; check if it exists first, if not create it

<br>

*Prompt 2 Chain-of-Thought*

Name the image after the df['sandwich_name'] column
  
<br>
  
*Prompt 3 Chain-of-Thought*

There is a rate limit, i can only submit two images per minute



In [ ]:
import os
import shutil
import time
import re
from imagepig import ImagePig

# Define the directory for saving processed images
# output_dir = "processed_images_new_run"

# Check if the directory exists and delete all files inside
# if os.path.exists(output_dir):
#     for file in os.listdir(output_dir):
#         file_path = os.path.join(output_dir, file)
#         if os.path.isfile(file_path):
#             os.remove(file_path)
#         elif os.path.isdir(file_path):
#             shutil.rmtree(file_path)  # If there are subdirectories, delete them

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get PIG API key
pig_api_key = os.getenv("PIG_API_KEY")

# Initialize ImagePig API
imagepig = ImagePig(pig_api_key)

# Function to sanitize filenames
def sanitize_filename(name):
    """Sanitize sandwich name to create a valid filename."""
    name = re.sub(r'[^\w\s-]', '', name)  # Remove special characters
    name = name.replace(" ", "_")  # Replace spaces with underscores
    return name.lower()  # Convert to lowercase for consistency

# Loop through each image URL in the 'image' column and process it
image_paths = []  # List to store paths of the processed images
for idx, (img_url, sandwich_name) in enumerate(zip(merged_df['image_final'], merged_df['sandwich'])):
    try:
        # Process the image using ImagePig
        result = imagepig.cutout(img_url)
        
        # Define the output filename using the sanitized sandwich name
        sanitized_name = sanitize_filename(sandwich_name)
        output_filename = f"{sanitized_name}.png"
        output_path = os.path.join(output_dir, output_filename)
        
        # Save the result to a file
        result.save(output_path)
        
        print(f"Processed and saved: {output_filename}")
        
        # Store the output path in the list
        image_paths.append(output_path)

        # If we've processed two images, sleep for a minute to respect the rate limit
        if (idx + 1) % 2 == 0:
            print("Rate limit reached. Sleeping for 60 seconds...")
            time.sleep(60)  # Sleep for 60 seconds (1 minute)
            
    except Exception as e:
        print(f"Failed to process image at {img_url}: {e}")
        image_paths.append(None)  # In case of failure, append None

# Add the image paths back to the DataFrame in the 'images_no_bg' column
merged_df['images_no_bg'] = image_paths


Processed and saved: chorizo_egg_sandwich.png
Processed and saved: smoked_salmon_sandwich.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: becl.png
Processed and saved: the_newhouse.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: shake_shack_breakfast_sandwich.png
Processed and saved: the_italian.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: the_casa.png
Processed and saved: egg_potato_and_cheese_hero.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: the_soho.png
Processed and saved: the_bomb.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: hlt.png
Processed and saved: crispy_chickn.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: scuttlebutt.png
Processed and saved: collard_greens_sandwich_on_focaccia.png
Rate limit reached. Sleeping for 60 seconds...
Processed and saved: french_onion_sandwich.png
Processed and saved: vegan_bbq_pork_bao_bun.png
R

In [ ]:
locations = []
for index, row in geo_cat_df.iterrows():
    location = {
        'sandwich_name': row['sandwich_name'],
        'restaurant': row['restaurant'],
        'lat': row['latitude'],
        'lng': row['longitude'],
        'img': row['image_url'],
        'address': row['address'],
        'price': row['price'],
        'description': row['description'],
        'website': row['website'],
        'img_no_bg': row['images_no_bg']
    }
    locations.append(location)